## Introduction to Iceberg Architecture

In [1]:
!pip3 install -r requirements.txt

  Cloning https://github.com/fastforwardlabs/cmlbootstrap to /tmp/pip-install-3p8kj1nh/cmlbootstrap_ad2cdb70864847688f99f6fd19ceda60
  Running command git clone -q https://github.com/fastforwardlabs/cmlbootstrap /tmp/pip-install-3p8kj1nh/cmlbootstrap_ad2cdb70864847688f99f6fd19ceda60


#### Launching a Spark Session with Iceberg

In [2]:
import cml.data_v1 as cmldata

CONNECTION_NAME = "go01-aw-dl"
conn = cmldata.get_connection(CONNECTION_NAME)
spark = conn.get_spark_session()

# Sample usage to run query through spark
EXAMPLE_SQL_QUERY = "show databases"
spark.sql(EXAMPLE_SQL_QUERY).show()

Setting spark.hadoop.yarn.resourcemanager.principal to pauldefusco
Hive Session ID = 3baef857-6eda-4b36-a35d-7b641ddb3bd7


+--------------------+
|           namespace|
+--------------------+
|         01_car_data|
|           01_car_dw|
|      adash_car_data|
|             airline|
|          airline_dw|
|            airlines|
|        airlines_csv|
|       airlines_csv1|
|   airlines_csv_vish|
|    airlines_iceberg|
|   airlines_iceberg1|
|airlines_iceberg_...|
|airlines_iceberg_...|
|      airlines_mjain|
|          airquality|
|          atlas_demo|
|            bankdemo|
|          bca_jps_l0|
|          bca_jps_l1|
|              bhagan|
+--------------------+
only showing top 20 rows



In [3]:
spark.sparkContext.getConf().getAll()

[('spark.driver.port', '40901'),
 ('spark.eventLog.enabled', 'true'),
 ('spark.repl.local.jars',
  'file:///runtime-addons/spark320-17-hf1-6xa3lk/opt/spark/optional-lib/iceberg-spark-runtime-3.2_2.12-0.14.1.1.17.7215.0-31.jar'),
 ('spark.network.crypto.enabled', 'true'),
 ('spark.sql.hive.hwc.execution.mode', 'spark'),
 ('spark.kerberos.renewal.credentials', 'ccache'),
 ('spark.app.startTime', '1684563308982'),
 ('spark.sql.catalog.spark_catalog',
  'org.apache.iceberg.spark.SparkSessionCatalog'),
 ('spark.dynamicAllocation.maxExecutors', '49'),
 ('spark.eventLog.dir', 'file:///sparkeventlogs'),
 ('spark.hadoop.yarn.resourcemanager.principal', 'pauldefusco'),
 ('spark.kubernetes.driver.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict',
  'false'),
 ('spark.ui.port', '20049'),
 ('spark.driver.bindAddress', '100.100.5.100'),
 ('spark.yarn.access.hadoopFileSystems',
  's3a://go01-demo/warehouse/tablespace/external/hive'),
 ('spark.sql.extensions',
  'com.qubole.spark.hiveacid.Hiv

### Iceberg Architecture

![alt text](../img/iceberg-metadata.png)

#### Iceberg Catalog

Iceberg comes with catalogs that enable SQL commands to manage tables and load them by name. Catalogs are configured using properties under spark.sql.catalog.(catalog_name).

In [4]:
# Show catalog and database
spark.sql("SHOW CURRENT NAMESPACE").show()

+-------------+---------+
|      catalog|namespace|
+-------------+---------+
|spark_catalog|  default|
+-------------+---------+



In [5]:
# Create a new database
#spark.sql("DROP DATABASE IF EXISTS spark_catalog.lakehouse")
spark.sql("CREATE DATABASE IF NOT EXISTS spark_catalog.lakehouse")
spark.sql("USE spark_catalog.lakehouse")

DataFrame[]

In [6]:
# Show catalog and database
spark.sql("SHOW CURRENT NAMESPACE").show()

+-------------+---------+
|      catalog|namespace|
+-------------+---------+
|spark_catalog|lakehouse|
+-------------+---------+



#### Create an Iceberg Table with Spark SQL

In [7]:
spark.sql("DROP TABLE IF EXISTS lakehouse.customer_table PURGE")

DataFrame[]

In [8]:
spark.sql("CREATE TABLE IF NOT EXISTS customer_table (id BIGINT, state STRING, country STRING, dob TIMESTAMP) USING iceberg PARTITIONED BY ( hours(dob))")

DataFrame[]

#### Verify that a Metadata JSON file has been created under the Metadata directory

In [39]:
metadata_path = "warehouse/tablespace/external/hive/lakehouse.db/customer_table"

In [21]:
import boto3

s3 = boto3.resource('s3')
my_bucket = s3.Bucket("go01-demo")

for object_summary in my_bucket.objects.filter(Prefix=metadata_path):
    #print(object_summary.key)
    metadata_file = object_summary.key
    
print("Metadata File Path: {}".format(metadata_file))

Metadata File Path: warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00000-4f5a53c7-8c79-4dd4-967c-ffa35855f838.metadata.json


In [18]:
import pandas as pd
spark.read.option("multiline","true").json("s3a://go01-demo/" + "warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata").toPandas()

,current-schema-id,current-snapshot-id,default-sort-order-id,default-spec-id,format-version,last-column-id,last-partition-id,last-updated-ms,location,metadata-log,partition-spec,partition-specs,properties,schema,schemas,snapshot-log,snapshots,sort-orders,table-uuid
0,0,-1,0,0,1,4,1000,1684536939127,s3a://go01-demo/warehouse/tablespace/external/...,[],"[(1000, dob_hour, 4, hour)]","[([Row(field-id=1000, name='dob_hour', source-...","(pauldefusco,)","([(1, id, False, long), (2, state, False, stri...","[([Row(id=1, name='id', required=False, type='...",[],[],"[([], 0)]",07c4aa75-1381-4887-930a-1acf7e10b43d


![alt text](../img/s3_metadata.png)

#### Notice that no snapshots or other files have been created as data has not yet been inserted.

In [28]:
spark.sql("SELECT * FROM lakehouse.customer_table.history").show()

+---------------+-----------+---------+-------------------+
|made_current_at|snapshot_id|parent_id|is_current_ancestor|
+---------------+-----------+---------+-------------------+
+---------------+-----------+---------+-------------------+



In [29]:
spark.sql("SELECT * FROM lakehouse.customer_table.snapshots;").show()

+------------+-----------+---------+---------+-------------+-------+
|committed_at|snapshot_id|parent_id|operation|manifest_list|summary|
+------------+-----------+---------+---------+-------------+-------+
+------------+-----------+---------+---------+-------------+-------+



In [30]:
spark.sql("SELECT * FROM lakehouse.customer_table.files;").show()

+-------+---------+-----------+-------+---------+------------+------------------+------------+------------+-----------------+----------------+------------+------------+------------+-------------+------------+-------------+
|content|file_path|file_format|spec_id|partition|record_count|file_size_in_bytes|column_sizes|value_counts|null_value_counts|nan_value_counts|lower_bounds|upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|
+-------+---------+-----------+-------+---------+------------+------------------+------------+------------+-----------------+----------------+------------+------------+------------+-------------+------------+-------------+
+-------+---------+-----------+-------+---------+------------+------------------+------------+------------+-----------------+----------------+------------+------------+------------+-------------+------------+-------------+



In [31]:
spark.sql("SELECT * FROM lakehouse.customer_table.manifests;").show()

+-------+----+------+-----------------+-----------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
|content|path|length|partition_spec_id|added_snapshot_id|added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries|
+-------+----+------+-----------------+-----------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
+-------+----+------+-----------------+-----------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+



In [32]:
spark.sql("SELECT * FROM lakehouse.customer_table.all_data_files;").show()

+-------+---------+-----------+-------+---------+------------+------------------+------------+------------+-----------------+----------------+------------+------------+------------+-------------+------------+-------------+
|content|file_path|file_format|spec_id|partition|record_count|file_size_in_bytes|column_sizes|value_counts|null_value_counts|nan_value_counts|lower_bounds|upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|
+-------+---------+-----------+-------+---------+------------+------------------+------------+------------+-----------------+----------------+------------+------------+------------+-------------+------------+-------------+
+-------+---------+-----------+-------+---------+------------+------------------+------------+------------+-----------------+----------------+------------+------------+------------+-------------+------------+-------------+



In [33]:
spark.sql("SELECT * FROM lakehouse.customer_table.all_manifests;").show()

+-------+----+------+-----------------+-----------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+---------------------+
|content|path|length|partition_spec_id|added_snapshot_id|added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries|reference_snapshot_id|
+-------+----+------+-----------------+-----------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+---------------------+
+-------+----+------+-----------------+-----------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+-------

### Table Insert

In [34]:
from pyspark.sql.functions import date_format

In [35]:
spark.sql("INSERT INTO lakehouse.customer_table VALUES (1, 'CA', 'USA', cast(date_format('2000-01-01 00:00:00', 'yyyy-MM-dd HH:mm:ss') as timestamp))")

DataFrame[]

#### Data has been added to the data folder

In [36]:
QUERY = "select h.made_current_at,\
            s.operation,\
            h.snapshot_id,\
            h.is_current_ancestor,\
            s.summary['spark.app.id']\
        from lakehouse.customer_table.history h\
        join lakehouse.customer_table.snapshots s\
            on h.snapshot_id = s.snapshot_id\
            order by made_current_at;"

In [37]:
spark.sql(QUERY).toPandas()

,made_current_at,operation,snapshot_id,is_current_ancestor,summary[spark.app.id]


#### Notice there are now two json files and two avro files. 

The second json file reflects the new table version after the insert. Now, a new table read operation will point to this new file and ignore the previous one.

The avro file with the "snap" prefix is the manifest list. The other avro file created is the corresponding manifest file.

In [45]:
s3 = boto3.resource('s3')
my_bucket = s3.Bucket("go01-demo")

metadata_file_list = []

print("Current Metadata Files: \n")
for object_summary in my_bucket.objects.filter(Prefix=metadata_path+"/metadata"):
    print(object_summary.key +"\n")
    metadata_file_list.append(object_summary.key)

Current Metadata Files: 

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00000-4f5a53c7-8c79-4dd4-967c-ffa35855f838.metadata.json

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00001-b19f559e-3ccf-4a57-9fd7-ae1cf48ce3f0.metadata.json

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2-m0.avro

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-3986362509610264284-1-385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2.avro



Showing Metadata Files (JSON)

In [46]:
import pandas as pd

print("Showing " + metadata_file_list[0])
spark.read.option("multiline","true").json("s3a://go01-demo/" + metadata_file_list[0]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00000-4f5a53c7-8c79-4dd4-967c-ffa35855f838.metadata.json


,current-schema-id,current-snapshot-id,default-sort-order-id,default-spec-id,format-version,last-column-id,last-partition-id,last-updated-ms,location,metadata-log,partition-spec,partition-specs,properties,schema,schemas,snapshot-log,snapshots,sort-orders,table-uuid
0,0,-1,0,0,1,4,1000,1684536939127,s3a://go01-demo/warehouse/tablespace/external/...,[],"[(1000, dob_hour, 4, hour)]","[([Row(field-id=1000, name='dob_hour', source-...","(pauldefusco,)","([(1, id, False, long), (2, state, False, stri...","[([Row(id=1, name='id', required=False, type='...",[],[],"[([], 0)]",07c4aa75-1381-4887-930a-1acf7e10b43d


In [47]:
print("Showing " + metadata_file_list[1])
spark.read.option("multiline","true").json("s3a://go01-demo/" + metadata_file_list[1]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00001-b19f559e-3ccf-4a57-9fd7-ae1cf48ce3f0.metadata.json


,current-schema-id,current-snapshot-id,default-sort-order-id,default-spec-id,format-version,last-column-id,last-partition-id,last-updated-ms,location,metadata-log,partition-spec,partition-specs,properties,refs,schema,schemas,snapshot-log,snapshots,sort-orders,table-uuid
0,0,3986362509610264284,0,0,1,4,1000,1684537347083,s3a://go01-demo/warehouse/tablespace/external/...,[(s3a://go01-demo/warehouse/tablespace/externa...,"[(1000, dob_hour, 4, hour)]","[([Row(field-id=1000, name='dob_hour', source-...","(pauldefusco,)","((3986362509610264284, branch),)","([(1, id, False, long), (2, state, False, stri...","[([Row(id=1, name='id', required=False, type='...","[(3986362509610264284, 1684537347083)]",[(s3a://go01-demo/warehouse/tablespace/externa...,"[([], 0)]",07c4aa75-1381-4887-930a-1acf7e10b43d


Showing Manifest Lists (AVRO - prefixed by "SNAP")

In [48]:
print("Showing " + metadata_file_list[3])
spark.read.format("avro").load("s3a://go01-demo/" + metadata_file_list[3]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-3986362509610264284-1-385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2.avro


,manifest_path,manifest_length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,partitions,added_rows_count,existing_rows_count,deleted_rows_count
0,s3a://go01-demo/warehouse/tablespace/external/...,6183,0,3986362509610264284,1,0,0,"[(False, False, [56, 3, 4, 0], [56, 3, 4, 0])]",1,0,0


Showing Manifest Files (Avro) i.e. list of table partitions mapped to snapshot ID

In [49]:
print("Showing " + metadata_file_list[2])
spark.read.format("avro").load("s3a://go01-demo/" + metadata_file_list[2]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2-m0.avro


,status,snapshot_id,data_file
0,1,3986362509610264284,(s3a://go01-demo/warehouse/tablespace/external...


### Table Update

Create a staging table

In [50]:
spark.sql("DROP TABLE IF EXISTS lakehouse.staging PURGE")

DataFrame[]

In [51]:
spark.sql("CREATE TABLE IF NOT EXISTS lakehouse.staging\
            (id BIGINT, state STRING, country STRING, dob TIMESTAMP)\
            USING iceberg\
            PARTITIONED BY ( hours(dob))")

DataFrame[]

In [52]:
spark.sql("INSERT INTO lakehouse.staging\
            VALUES (1, 'ID', 'USA', cast(date_format('2000-01-01 00:00:00', 'yyyy-MM-dd HH:mm:ss') as timestamp)),\
            (2, 'CA', 'USA', cast(date_format('2000-01-03 00:10:00', 'yyyy-MM-dd HH:mm:ss') as timestamp)),\
            (3, 'AZ', 'USA', cast(date_format('2000-01-04 00:01:00', 'yyyy-MM-dd HH:mm:ss') as timestamp)),\
            (4, 'NV', 'USA', cast(date_format('2000-01-02 00:02:00', 'yyyy-MM-dd HH:mm:ss') as timestamp)),\
            (5, 'OR', 'USA', cast(date_format('2000-01-03 00:03:00', 'yyyy-MM-dd HH:mm:ss') as timestamp)),\
            (10, 'WA', 'USA', cast(date_format('2000-01-04 00:03:00', 'yyyy-MM-dd HH:mm:ss') as timestamp)),\
            (3, 'UT', 'USA', cast(date_format('2000-01-01 00:04:00', 'yyyy-MM-dd HH:mm:ss') as timestamp)),\
            (11, 'CO', 'USA', cast(date_format('2000-01-01 00:03:00', 'yyyy-MM-dd HH:mm:ss') as timestamp)),\
            (6, 'CO', 'USA', cast(date_format('2000-01-01 00:03:00', 'yyyy-MM-dd HH:mm:ss') as timestamp))")

DataFrame[]

Merge Into Customers Table

In [55]:
spark.sql("MERGE INTO lakehouse.customer_table\
            USING (SELECT * FROM lakehouse.staging) a\
            ON customer_table.id = a.id\
            WHEN MATCHED THEN UPDATE SET customer_table.state = a.state\
            WHEN NOT MATCHED THEN INSERT *")

DataFrame[]

In [56]:
spark.sql("SELECT * FROM lakehouse.customer_table.snapshots;").toPandas()

,committed_at,snapshot_id,parent_id,operation,manifest_list,summary
0,2023-05-19 23:02:27.083,3986362509610264284,NaN,append,s3a://go01-demo/warehouse/tablespace/external/...,{'spark.app.id': 'spark-application-1684536915...
1,2023-05-19 23:06:44.014,6274129147237535833,3.986363e+18,overwrite,s3a://go01-demo/warehouse/tablespace/external/...,"{'added-data-files': '4', 'total-equality-dele..."


In [57]:
spark.sql("SELECT * FROM lakehouse.customer_table.manifests;").toPandas()

,content,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,added_delete_files_count,existing_delete_files_count,deleted_delete_files_count,partition_summaries
0,0,s3a://go01-demo/warehouse/tablespace/external/...,6365,0,6274129147237535833,4,0,0,0,0,0,"[(False, False, 2000-01-01-00, 2000-01-04-00)]"
1,0,s3a://go01-demo/warehouse/tablespace/external/...,6184,0,6274129147237535833,0,0,1,0,0,0,"[(False, False, 2000-01-01-00, 2000-01-01-00)]"


In [58]:
spark.sql("SELECT * FROM lakehouse.customer_table.all_data_files;").toPandas()

,content,file_path,file_format,spec_id,partition,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id
0,0,s3a://go01-demo/warehouse/tablespace/external/...,PARQUET,0,"(262968,)",1,1106,"{1: 33, 2: 31, 3: 32, 4: 39}","{1: 1, 2: 1, 3: 1, 4: 1}","{1: 0, 2: 0, 3: 0, 4: 0}",{},"{1: [1, 0, 0, 0, 0, 0, 0, 0], 2: [67, 65], 3: ...","{1: [1, 0, 0, 0, 0, 0, 0, 0], 2: [67, 65], 3: ...",None,[4],None,0
1,0,s3a://go01-demo/warehouse/tablespace/external/...,PARQUET,0,"(262968,)",4,1277,"{1: 57, 2: 74, 3: 61, 4: 80}","{1: 4, 2: 4, 3: 4, 4: 4}","{1: 0, 2: 0, 3: 0, 4: 0}",{},"{1: [1, 0, 0, 0, 0, 0, 0, 0], 2: [67, 79], 3: ...","{1: [11, 0, 0, 0, 0, 0, 0, 0], 2: [85, 84], 3:...",None,[4],None,0
2,0,s3a://go01-demo/warehouse/tablespace/external/...,PARQUET,0,"(262992,)",1,1130,"{1: 39, 2: 37, 3: 38, 4: 39}","{1: 1, 2: 1, 3: 1, 4: 1}","{1: 0, 2: 0, 3: 0, 4: 0}",{},"{1: [4, 0, 0, 0, 0, 0, 0, 0], 2: [78, 86], 3: ...","{1: [4, 0, 0, 0, 0, 0, 0, 0], 2: [78, 86], 3: ...",None,[4],None,0
3,0,s3a://go01-demo/warehouse/tablespace/external/...,PARQUET,0,"(263016,)",2,1176,"{1: 46, 2: 43, 3: 61, 4: 47}","{1: 2, 2: 2, 3: 2, 4: 2}","{1: 0, 2: 0, 3: 0, 4: 0}",{},"{1: [2, 0, 0, 0, 0, 0, 0, 0], 2: [67, 65], 3: ...","{1: [5, 0, 0, 0, 0, 0, 0, 0], 2: [79, 82], 3: ...",None,[4],None,0
4,0,s3a://go01-demo/warehouse/tablespace/external/...,PARQUET,0,"(263040,)",2,1177,"{1: 47, 2: 43, 3: 61, 4: 47}","{1: 2, 2: 2, 3: 2, 4: 2}","{1: 0, 2: 0, 3: 0, 4: 0}",{},"{1: [3, 0, 0, 0, 0, 0, 0, 0], 2: [65, 90], 3: ...","{1: [10, 0, 0, 0, 0, 0, 0, 0], 2: [87, 65], 3:...",None,[4],None,0


#### There is a new metadata file (json) prefixed by 0002.

#### There is a new manifest list file (avro) prefixed by "snap"

#### There is a new manifest file (avro)

In [61]:
s3 = boto3.resource('s3')
my_bucket = s3.Bucket("go01-demo")

metadata_file_list = []

print("Current Metadata Files: \n")

for object_summary in my_bucket.objects.filter(Prefix=metadata_path+"/metadata"):
    #print(object_summary.key +"\n")
    metadata_file_list.append(object_summary.key)
    
metadata_file_list

Current Metadata Files: 

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00000-4f5a53c7-8c79-4dd4-967c-ffa35855f838.metadata.json

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00001-b19f559e-3ccf-4a57-9fd7-ae1cf48ce3f0.metadata.json

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00002-30665850-e97b-46dd-a4f1-2dee67bbe972.metadata.json

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/021df821-94c9-424f-af1a-51fa31667010-m0.avro

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/021df821-94c9-424f-af1a-51fa31667010-m1.avro

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2-m0.avro

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-3986362509610264284-1-385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2.avro

warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-62741291472

['warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00000-4f5a53c7-8c79-4dd4-967c-ffa35855f838.metadata.json',
 'warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00001-b19f559e-3ccf-4a57-9fd7-ae1cf48ce3f0.metadata.json',
 'warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00002-30665850-e97b-46dd-a4f1-2dee67bbe972.metadata.json',
 'warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/021df821-94c9-424f-af1a-51fa31667010-m0.avro',
 'warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/021df821-94c9-424f-af1a-51fa31667010-m1.avro',
 'warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2-m0.avro',
 'warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-3986362509610264284-1-385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2.avro',
 'warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-62741291472375

Showing Metadata Files (JSON)

In [62]:
print("Showing " + metadata_file_list[0])
spark.read.option("multiline","true").json("s3a://go01-demo/" + metadata_file_list[0]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00000-4f5a53c7-8c79-4dd4-967c-ffa35855f838.metadata.json


,current-schema-id,current-snapshot-id,default-sort-order-id,default-spec-id,format-version,last-column-id,last-partition-id,last-updated-ms,location,metadata-log,partition-spec,partition-specs,properties,schema,schemas,snapshot-log,snapshots,sort-orders,table-uuid
0,0,-1,0,0,1,4,1000,1684536939127,s3a://go01-demo/warehouse/tablespace/external/...,[],"[(1000, dob_hour, 4, hour)]","[([Row(field-id=1000, name='dob_hour', source-...","(pauldefusco,)","([(1, id, False, long), (2, state, False, stri...","[([Row(id=1, name='id', required=False, type='...",[],[],"[([], 0)]",07c4aa75-1381-4887-930a-1acf7e10b43d


In [63]:
print("Showing " + metadata_file_list[1])
spark.read.option("multiline","true").json("s3a://go01-demo/" + metadata_file_list[1]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00001-b19f559e-3ccf-4a57-9fd7-ae1cf48ce3f0.metadata.json


,current-schema-id,current-snapshot-id,default-sort-order-id,default-spec-id,format-version,last-column-id,last-partition-id,last-updated-ms,location,metadata-log,partition-spec,partition-specs,properties,refs,schema,schemas,snapshot-log,snapshots,sort-orders,table-uuid
0,0,3986362509610264284,0,0,1,4,1000,1684537347083,s3a://go01-demo/warehouse/tablespace/external/...,[(s3a://go01-demo/warehouse/tablespace/externa...,"[(1000, dob_hour, 4, hour)]","[([Row(field-id=1000, name='dob_hour', source-...","(pauldefusco,)","((3986362509610264284, branch),)","([(1, id, False, long), (2, state, False, stri...","[([Row(id=1, name='id', required=False, type='...","[(3986362509610264284, 1684537347083)]",[(s3a://go01-demo/warehouse/tablespace/externa...,"[([], 0)]",07c4aa75-1381-4887-930a-1acf7e10b43d


In [64]:
print("Showing " + metadata_file_list[2])
spark.read.option("multiline","true").json("s3a://go01-demo/" + metadata_file_list[2]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00002-30665850-e97b-46dd-a4f1-2dee67bbe972.metadata.json


,current-schema-id,current-snapshot-id,default-sort-order-id,default-spec-id,format-version,last-column-id,last-partition-id,last-updated-ms,location,metadata-log,partition-spec,partition-specs,properties,refs,schema,schemas,snapshot-log,snapshots,sort-orders,table-uuid
0,0,6274129147237535833,0,0,1,4,1000,1684537604014,s3a://go01-demo/warehouse/tablespace/external/...,[(s3a://go01-demo/warehouse/tablespace/externa...,"[(1000, dob_hour, 4, hour)]","[([Row(field-id=1000, name='dob_hour', source-...","(pauldefusco,)","((6274129147237535833, branch),)","([(1, id, False, long), (2, state, False, stri...","[([Row(id=1, name='id', required=False, type='...","[(3986362509610264284, 1684537347083), (627412...",[(s3a://go01-demo/warehouse/tablespace/externa...,"[([], 0)]",07c4aa75-1381-4887-930a-1acf7e10b43d


Showing Manifest Lists (AVRO - prefixed by "SNAP")

In [65]:
print("Showing " + metadata_file_list[6])
spark.read.format("avro").load("s3a://go01-demo/" + metadata_file_list[6]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-3986362509610264284-1-385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2.avro


,manifest_path,manifest_length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,partitions,added_rows_count,existing_rows_count,deleted_rows_count
0,s3a://go01-demo/warehouse/tablespace/external/...,6183,0,3986362509610264284,1,0,0,"[(False, False, [56, 3, 4, 0], [56, 3, 4, 0])]",1,0,0


In [66]:
print("Showing " + metadata_file_list[7])
spark.read.format("avro").load("s3a://go01-demo/" + metadata_file_list[7]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-6274129147237535833-1-021df821-94c9-424f-af1a-51fa31667010.avro


,manifest_path,manifest_length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,partitions,added_rows_count,existing_rows_count,deleted_rows_count
0,s3a://go01-demo/warehouse/tablespace/external/...,6365,0,6274129147237535833,4,0,0,"[(False, False, [56, 3, 4, 0], [128, 3, 4, 0])]",9,0,0
1,s3a://go01-demo/warehouse/tablespace/external/...,6184,0,6274129147237535833,0,0,1,"[(False, False, [56, 3, 4, 0], [56, 3, 4, 0])]",0,0,1


Showing Manifest Files (Avro) i.e. list of table partitions mapped to snapshot ID

In [67]:
print("Showing " + metadata_file_list[3])
spark.read.format("avro").load("s3a://go01-demo/" + metadata_file_list[3]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/021df821-94c9-424f-af1a-51fa31667010-m0.avro


,status,snapshot_id,data_file
0,2,6274129147237535833,(s3a://go01-demo/warehouse/tablespace/external...


In [68]:
print("Showing " + metadata_file_list[4])
spark.read.format("avro").load("s3a://go01-demo/" + metadata_file_list[4]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/021df821-94c9-424f-af1a-51fa31667010-m1.avro


,status,snapshot_id,data_file
0,1,6274129147237535833,(s3a://go01-demo/warehouse/tablespace/external...
1,1,6274129147237535833,(s3a://go01-demo/warehouse/tablespace/external...
2,1,6274129147237535833,(s3a://go01-demo/warehouse/tablespace/external...
3,1,6274129147237535833,(s3a://go01-demo/warehouse/tablespace/external...


In [69]:
print("Showing " + metadata_file_list[5])
spark.read.format("avro").load("s3a://go01-demo/" + metadata_file_list[5]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2-m0.avro


,status,snapshot_id,data_file
0,1,3986362509610264284,(s3a://go01-demo/warehouse/tablespace/external...


### Time Travel 

In [73]:
snapshots_df = spark.sql("SELECT * FROM lakehouse.customer_table.snapshots;")

In [74]:
first_snapshot = snapshots_df.select("snapshot_id").head(1)[0][0]

#### Validate that the output dataframe only includes one row per the original insert

In [76]:
spark.read\
    .option("snapshot-id", first_snapshot)\
    .format("iceberg")\
    .load("lakehouse.customer_table").toPandas()

,id,state,country,dob
0,1,CA,USA,2000-01-01


In [78]:
avro_tempdf = spark.read.format("avro").load("s3a://go01-demo/" + metadata_file_list[6]).toPandas()

In [80]:
avro_tempdf.columns

Index(['manifest_path', 'manifest_length', 'partition_spec_id',
       'added_snapshot_id', 'added_data_files_count',
       'existing_data_files_count', 'deleted_data_files_count', 'partitions',
       'added_rows_count', 'existing_rows_count', 'deleted_rows_count'],
      dtype='object')

In [79]:
avro_tempdf['partitions']

0    [(False, False, [56, 3, 4, 0], [56, 3, 4, 0])]
Name: partitions, dtype: object

In [81]:
avro_tempdf['added_rows_count']

0    1
Name: added_rows_count, dtype: int64

In [82]:
avro_tempdf['existing_rows_count']

0    0
Name: existing_rows_count, dtype: int64

In [84]:
avro_tempdf['added_data_files_count']

0    1
Name: added_data_files_count, dtype: int32

In [85]:
print("Showing " + metadata_file_list[2])
json_tempdf = spark.read.option("multiline","true").json("s3a://go01-demo/" + metadata_file_list[2]).toPandas()

Showing warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/00002-30665850-e97b-46dd-a4f1-2dee67bbe972.metadata.json


In [86]:
json_tempdf.columns

Index(['current-schema-id', 'current-snapshot-id', 'default-sort-order-id',
       'default-spec-id', 'format-version', 'last-column-id',
       'last-partition-id', 'last-updated-ms', 'location', 'metadata-log',
       'partition-spec', 'partition-specs', 'properties', 'refs', 'schema',
       'schemas', 'snapshot-log', 'snapshots', 'sort-orders', 'table-uuid'],
      dtype='object')

In [90]:
json_tempdf['current-schema-id']

0    0
Name: current-schema-id, dtype: int64

In [92]:
list(json_tempdf['snapshots'])

[[Row(manifest-list='s3a://go01-demo/warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-3986362509610264284-1-385f35b6-3a5a-4d30-a7a1-156fa6a8c7b2.avro', parent-snapshot-id=None, schema-id=0, snapshot-id=3986362509610264284, summary=Row(added-data-files='1', added-files-size='1106', added-records='1', changed-partition-count='1', deleted-data-files=None, deleted-records=None, operation='append', removed-files-size=None, spark.app.id='spark-application-1684536915149', total-data-files='1', total-delete-files='0', total-equality-deletes='0', total-files-size='1106', total-position-deletes='0', total-records='1'), timestamp-ms=1684537347083),
  Row(manifest-list='s3a://go01-demo/warehouse/tablespace/external/hive/lakehouse.db/customer_table/metadata/snap-6274129147237535833-1-021df821-94c9-424f-af1a-51fa31667010.avro', parent-snapshot-id=3986362509610264284, schema-id=0, snapshot-id=6274129147237535833, summary=Row(added-data-files='4', added-files-size='4760', a

In [93]:
json_tempdf['partition-spec']

0    [(1000, dob_hour, 4, hour)]
Name: partition-spec, dtype: object

In [98]:
list(json_tempdf['partition-specs'])

[[Row(fields=[Row(field-id=1000, name='dob_hour', source-id=4, transform='hour')], spec-id=0)]]

### Partition Evolution

Spark partitioning is a way to split the data into multiple partitions so that you can execute transformations on multiple partitions in parallel which allows completing the job faster. You can also write partitioned data into a file system (multiple sub-directories) for faster reads by downstream systems.

Spark has several partitioning methods to achieve parallelism, based on your need, you should choose which one to use.

Creating New Data to Test Partition Evolution

In [7]:
from pyspark.sql.types import LongType, IntegerType, StringType

import dbldatagen as dg

shuffle_partitions_requested = 20
device_population = 100000
data_rows = 20 * 1000000
#partitions_requested = 20

spark.conf.set("spark.sql.shuffle.partitions", shuffle_partitions_requested)

country_codes = [
    "CN", "US", "FR", "CA", "IN", "JM", "IE", "PK", "GB", "IL", "AU", 
    "SG", "ES", "GE", "MX", "ET", "SA", "LB", "NL", "IT"
]
#country_weights = [
#    1300, 365, 67, 38, 1300, 3, 7, 212, 67, 9, 25, 6, 47, 83, 
#    126, 109, 58, 8, 17,
#]

manufacturers = [
    "Delta corp", "Xyzzy Inc.", "Lakehouse Ltd", "Acme Corp", "Embanks Devices",
]

lines = ["delta", "xyzzy", "lakehouse", "gadget", "droid"]

testDataSpec = (
    dg.DataGenerator(spark, name="device_data_set", rows=data_rows) 
                     #,partitions=partitions_requested)
    .withIdOutput()
    # we'll use hash of the base field to generate the ids to
    # avoid a simple incrementing sequence
    .withColumn("internal_device_id", "long", minValue=0x1000000000000, 
                uniqueValues=device_population, omit=True, baseColumnType="hash",
    )
    # note for format strings, we must use "%lx" not "%x" as the
    # underlying value is a long
    .withColumn(
        "device_id", "string", format="0x%013x", baseColumn="internal_device_id"
    )
    # the device / user attributes will be the same for the same device id
    # so lets use the internal device id as the base column for these attribute
    .withColumn("country", "string", values=country_codes, #weights=country_weights, 
                baseColumn="internal_device_id")
    .withColumn("manufacturer", "string", values=manufacturers, 
                baseColumn="internal_device_id", )
    # use omit = True if you don't want a column to appear in the final output
    # but just want to use it as part of generation of another column
    .withColumn("line", "string", values=lines, baseColumn="manufacturer", 
                baseColumnType="hash", omit=True )
    .withColumn("model_ser", "integer", minValue=1, maxValue=11, baseColumn="device_id", 
                baseColumnType="hash", omit=True, )
    .withColumn("model_line", "string", expr="concat(line, '#', model_ser)", 
                baseColumn=["line", "model_ser"] )
    .withColumn("event_type", "string", 
                values=["activation", "deactivation", "plan change", "telecoms activity", 
                        "internet activity", "device error", ],
                random=True)
    .withColumn("event_ts", "timestamp", begin="2020-01-01 01:00:00", 
                end="2020-12-31 23:59:00", 
                interval="1 minute", random=True )
)

dfTestData = testDataSpec.build()

display(dfTestData)

INFO: Version : VersionInfo(major='0', minor='2', patch='1', release='', build='')


DataFrame[id: bigint, device_id: string, country: string, manufacturer: string, model_line: string, event_type: string, event_ts: timestamp]

In [8]:
dfTestData.head()

Row(id=0, device_id='0x100000001281d', country='US', manufacturer='Xyzzy Inc.', model_line='lakehouse#10', event_type='internet activity', event_ts=datetime.datetime(2020, 11, 4, 3, 44))

In [9]:
spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")

In [10]:
spark.sql("DROP TABLE IF EXISTS spark_catalog.lakehouse.partition_evol_tbl PURGE")

DataFrame[]

In [11]:
#dfTestData.groupBy("country").count().show()

In [12]:
#dfTestData.rdd.getNumPartitions()

Iceberg requires the data to be sorted according to the partition spec per task (Spark partition) in prior to write against partitioned table. This applies both Writing with SQL and Writing with DataFrames.

In [13]:
dfTestData.sortWithinPartitions("country").writeTo("spark_catalog.lakehouse.p_evol_tbl").partitionedBy("country").using("iceberg").create()#.append()#replace()#overwritePartitions()#create()

In [ ]:
#spark.sql("SELECT * FROM spark_catalog.lakehouse.part_evol_tbl.PARTITIONS").show()

In [ ]:
#spark.sql("SELECT * FROM spark_catalog.lakehouse.part_evol_tbl.files").show()

In [ ]:
#spark.sql("SELECT * FROM spark_catalog.lakehouse.part_evol_tbl.manifests").show()

In [ ]:
#spark.sql("SELECT * FROM spark_catalog.lakehouse.part_evol_tbl.all_manifests").show()

In [ ]:
#spark.sql("SELECT * FROM spark_catalog.lakehouse.part_evol_tbl.all_data_files").show()

In [ ]:
#spark.sql("SELECT * FROM spark_catalog.lakehouse.part_evol_tbl.snapshots").show()

Adding a partition field is a metadata operation and does not change any of the existing table data. New data will be written with the new partitioning, but existing data will remain in the old partition layout. Old data files will have null values for the new partition fields in metadata tables.

In [14]:
print("TABLE PARTITIONS BEFORE ALTER PARTITION STATEMENT: ")
spark.sql("SELECT * FROM spark_catalog.lakehouse.p_evol_tbl.PARTITIONS").show()

TABLE PARTITIONS BEFORE ALTER PARTITION STATEMENT: 
+---------+------------+----------+-------+
|partition|record_count|file_count|spec_id|
+---------+------------+----------+-------+
|     {CN}|     1000100|        10|      0|
|     {CA}|     1000301|        10|      0|
|     {IT}|      999401|        10|      0|
|     {IL}|     1000474|        10|      0|
|     {NL}|     1000498|        10|      0|
|     {AU}|     1000859|        10|      0|
|     {GB}|      999318|        10|      0|
|     {US}|     1000098|        10|      0|
|     {SG}|      999532|        10|      0|
|     {LB}|      998789|        10|      0|
|     {IN}|      999328|        10|      0|
|     {IE}|      998731|        10|      0|
|     {FR}|      999338|        10|      0|
|     {ES}|     1000299|        10|      0|
|     {GE}|      999322|        10|      0|
|     {MX}|     1001198|        10|      0|
|     {PK}|     1002144|        10|      0|
|     {JM}|     1000840|        10|      0|
|     {SA}|      998932|

In [15]:
print("ADD PARTITION BY EVENT TIMESTAMP MONTHS: ")
print("ALTER TABLE spark_catalog.lakehouse.p_evol_tbl ADD PARTITION FIELD months(event_ts)")
spark.sql("ALTER TABLE spark_catalog.lakehouse.p_evol_tbl ADD PARTITION FIELD months(event_ts)")
#spark.sql("ALTER TABLE spark_catalog.lakehouse.part_evol_tbl REPLACE PARTITION FIELD hours(dob) WITH state")
#spark.sql("ALTER TABLE prod.db.sample ADD PARTITION FIELD month")

#ALTER TABLE spark_catalog.lakehouse.part_evol_tbl ADD PARTITION FIELD days(event_ts)

ADD PARTITION BY BUCKETING DEVICE ID:
ALTER TABLE spark_catalog.lakehouse.p_evol_tbl ADD PARTITION FIELD months(event_ts)


DataFrame[]

In [16]:
print("TABLE PARTITIONS AFTER ALTER PARTITION STATEMENT: ")
spark.sql("SELECT * FROM spark_catalog.lakehouse.p_evol_tbl.PARTITIONS").show()

TABLE PARTITIONS AFTER ALTER PARTITION STATEMENT: 


+---------+------------+----------+-------+
|partition|record_count|file_count|spec_id|
+---------+------------+----------+-------+
|     {US}|     1000098|        10|      0|
|     {LB}|      998789|        10|      0|
|     {FR}|      999338|        10|      0|
|     {GE}|      999322|        10|      0|
|     {IN}|      999328|        10|      0|
|     {SG}|      999532|        10|      0|
|     {IE}|      998731|        10|      0|
|     {JM}|     1000840|        10|      0|
|     {CN}|     1000100|        10|      0|
|     {SA}|      998932|        10|      0|
|     {AU}|     1000859|        10|      0|
|     {IT}|      999401|        10|      0|
|     {ET}|     1000498|        10|      0|
|     {PK}|     1002144|        10|      0|
|     {GB}|      999318|        10|      0|
|     {MX}|     1001198|        10|      0|
|     {NL}|     1000498|        10|      0|
|     {CA}|     1000301|        10|      0|
|     {IL}|     1000474|        10|      0|
|     {ES}|     1000299|        

In [17]:
appendDf = dfTestData.sample(fraction=0.3, seed=3)

In [18]:
appendDf.dtypes

[('id', 'bigint'),
 ('device_id', 'string'),
 ('country', 'string'),
 ('manufacturer', 'string'),
 ('model_line', 'string'),
 ('event_type', 'string'),
 ('event_ts', 'timestamp')]

In [19]:
appendDf.rdd.getNumPartitions()

10

In [20]:
appendDf.show()

+---+---------------+-------+---------------+------------+-----------------+-------------------+
| id|      device_id|country|   manufacturer|  model_line|       event_type|           event_ts|
+---+---------------+-------+---------------+------------+-----------------+-------------------+
|  0|0x100000001281d|     US|     Xyzzy Inc.|lakehouse#10|internet activity|2020-11-04 03:44:00|
|  4|0x1000000003674|     SA|     Xyzzy Inc.| lakehouse#6|       activation|2020-07-05 01:45:00|
|  5|0x100000001492c|     IN|Embanks Devices| lakehouse#7|telecoms activity|2020-01-24 22:31:00|
| 17|0x1000000010aa2|     MX|Embanks Devices| lakehouse#1|     deactivation|2020-09-29 01:01:00|
| 20|0x1000000001f27|     SG|     Xyzzy Inc.|lakehouse#11|      plan change|2020-10-26 00:21:00|
| 23|0x10000000039b4|     GB|      Acme Corp|    droid#10|     deactivation|2020-11-11 02:00:00|
| 28|0x1000000003a10|     CN|     Delta corp|    gadget#1|       activation|2020-10-28 03:30:00|
| 29|0x1000000014cd0|     SA| 

In [21]:
appendDf.sortWithinPartitions("country").show()

+----+---------------+-------+------------+----------+-----------------+-------------------+
|  id|      device_id|country|manufacturer|model_line|       event_type|           event_ts|
+----+---------------+-------+------------+----------+-----------------+-------------------+
| 132|0x1000000008146|     AU|  Delta corp|  gadget#2|     device error|2020-04-24 10:00:00|
| 150|0x100000001540e|     AU|  Delta corp| gadget#11|internet activity|2020-10-11 06:21:00|
| 217|0x100000000582e|     AU|  Delta corp|  gadget#2|telecoms activity|2020-04-11 14:24:00|
| 279|0x10000000183b6|     AU|  Delta corp| gadget#10|      plan change|2020-03-31 04:31:00|
| 368|0x100000000c2d2|     AU|  Delta corp|  gadget#1|       activation|2020-03-25 11:11:00|
| 489|0x10000000185d2|     AU|  Delta corp|  gadget#8|      plan change|2020-12-11 13:21:00|
| 515|0x100000001729a|     AU|  Delta corp|  gadget#3|telecoms activity|2020-04-12 14:19:00|
| 804|0x100000000d1be|     AU|  Delta corp|  gadget#2|      plan chang

In [22]:
#appendDf.sortWithinPartitions("country", "month(event_ts)").show()

In [23]:
appendDf.sortWithinPartitions("country").writeTo("spark_catalog.lakehouse.p_evol_tbl").using("iceberg").append() #.append()#replace()#overwritePartitions()#create()

In [24]:
print("TABLE PARTITIONS AFTER APPEND: ")
spark.sql("SELECT * FROM spark_catalog.lakehouse.p_evol_tbl.PARTITIONS").show(100)

TABLE PARTITIONS AFTER APPEND: 
+----------+------------+----------+-------+
| partition|record_count|file_count|spec_id|
+----------+------------+----------+-------+
| {MX, 610}|       24539|        10|      1|
| {CA, 606}|       25261|        10|      1|
| {ES, 603}|       24582|        10|      1|
| {MX, 601}|       23816|        10|      1|
|{SG, null}|      999532|        10|      0|
| {FR, 600}|       25141|        10|      1|
| {IE, 600}|       25480|        10|      1|
| {IE, 603}|       24541|        10|      1|
| {SG, 600}|       25147|        10|      1|
| {SG, 604}|       25366|        10|      1|
| {IL, 601}|       23763|        10|      1|
| {MX, 611}|       25176|        10|      1|
| {IE, 605}|       24326|        10|      1|
| {LB, 607}|       25535|        10|      1|
| {IN, 611}|       25312|        10|      1|
| {JM, 601}|       24042|        10|      1|
|{IE, null}|      998731|        10|      0|
| {GE, 611}|       25223|        10|      1|
| {CA, 608}|       2466

Dropping a partition field is a metadata operation and does not change any of the existing table data. New data will be written with the new partitioning, but existing data will remain in the old partition layout.

In [74]:
spark.sql("ALTER TABLE spark_catalog.lakehouse.part_evol_tbl DROP PARTITION FIELD bucket(16, device_id)")

DataFrame[]

In [75]:
print("TABLE PARTITIONS AFTER ALTER PARTITION STATEMENT: ")
spark.sql("SELECT * FROM spark_catalog.lakehouse.part_evol_tbl.PARTITIONS").show()

CAR SALES TABLE PARTITIONS AFTER ALTER PARTITION STATEMENT: 


+----------+------------+----------+-------+
| partition|record_count|file_count|spec_id|
+----------+------------+----------+-------+
|{CN, null}|    10000790|        10|      0|
|{IT, null}|    10002099|        10|      0|
|{US, null}|     9998801|        10|      0|
|{AU, null}|     9997695|        10|      0|
|{GE, null}|    10005212|        10|      0|
|{PK, null}|    10002237|        10|      0|
|{ES, null}|    10001299|        10|      0|
|{SA, null}|     9998077|        10|      0|
|{ET, null}|    10001822|        10|      0|
|{NL, null}|     9998304|        10|      0|
|{IN, null}|    10004181|        10|      0|
|{MX, null}|    10002210|        10|      0|
|{GB, null}|     9993405|        10|      0|
|{IE, null}|     9998066|        10|      0|
|{CA, null}|     9997659|        10|      0|
|{LB, null}|     9996707|        10|      0|
|{JM, null}|    10000921|        10|      0|
|{IL, null}|     9998131|        10|      0|
|{FR, null}|    10003914|        10|      0|
|{SG, null

##### Only json files have been added (one per each time you repartitioned) but Avro files have stayed the same

In [57]:
s3 = boto3.resource('s3')
my_bucket = s3.Bucket("go01-demo")

metadata_file_list = []

print("Current Metadata Files: \n")

for object_summary in my_bucket.objects.filter(Prefix=metadata_path+"/metadata"):
    #print(object_summary.key +"\n")
    metadata_file_list.append(object_summary.key)
    
metadata_file_list

NameError: name 'boto3' is not defined

In [ ]:
spark.sql("CREATE TABLE IF NOT EXISTS customer_table (id BIGINT, state STRING, country STRING, dob TIMESTAMP) USING iceberg PARTITIONED BY ( hours(dob))")

In [110]:
spark.sql("SELECT HOUR(dob) FROM spark_catalog.lakehouse.customer_table").show()

+---------+
|hour(dob)|
+---------+
|        0|
|        0|
|        0|
|        0|
|        0|
|        0|
|        0|
|        0|
|        0|
+---------+



In [107]:
spark.sql("SELECT DAY(dob) FROM spark_catalog.lakehouse.customer_table").show()

+--------+
|day(dob)|
+--------+
|       1|
|       1|
|       1|
|       1|
|       2|
|       3|
|       3|
|       4|
|       4|
+--------+



### Dropping Tables

In [ ]:
spark.sql("DROP TABLE IF EXISTS lakehouse.staging")

Validate that the metadata folder is now empty but the data folder still retains parquet files.

![alt text](../img/s3_droptable_1.png)

![alt text](../img/s3_droptable_2.png)

![alt text](../img/s3_droptable_3.png)

In [ ]:
spark.sql("ALTER TABLE lakehouse.customers_table\
            SET TBLPROPERTIES ('format-version' = '2')")

In [26]:
s3 = boto3.resource('s3')
my_bucket = s3.Bucket("go01-demo")

metadata_file_list = []

for object_summary in my_bucket.objects.filter(Prefix=metadata_path):
    print(object_summary.key +"\n")
    metadata_file_list.append(object_summary.key)

warehouse/tablespace/external/hive/lakehouse_catalog.db/customers_table/metadata/00000-8f9c2e19-d640-404e-8fc8-c4287b667740.metadata.json



In [27]:
print("Showing " + metadata_file_list[3])
spark.read.option("multiline","true").json("s3a://go01-demo/" + metadata_file_list[3]).toPandas()

IndexError: list index out of range